# Rice Disease Advisory System

## Objective

Generate disease recommendations after model prediction.

In [1]:
import json
import sys

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from src.config import *

In [10]:
from PIL import Image
from torchvision import transforms
import random

# Select a random image
all_images = list(DATASET_PATH.rglob("*.jpg"))
image_path = random.choice(all_images)

image = Image.open(image_path).convert("RGB")

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

input_tensor = transform(image).unsqueeze(0).to(DEVICE)

print(input_tensor.shape)

torch.Size([1, 3, 224, 224])


In [13]:
with torch.no_grad():

    output = model(input_tensor)

    probabilities = torch.softmax(output, dim=1)

    confidence, predicted = torch.max(probabilities, 1)

predicted_class = CLASS_NAMES[predicted.item()]
confidence = confidence.item()

print("Prediction :", predicted_class)
print(f"Confidence : {confidence*100:.2f}%")

Prediction : Bacterialblight
Confidence : 98.48%


In [11]:
knowledge_path = PROJECT_ROOT / "knowledge_base" / "rice_disease_info.json"

with open(knowledge_path, "r") as file:
    disease_db = json.load(file)

print("Knowledge Base Loaded Successfully")

Knowledge Base Loaded Successfully


In [15]:
from src.model import build_model
from src.utils import load_checkpoint

# Build model
model = build_model(pretrained=False)

# Load trained weights
model = load_checkpoint(
    model,
    MODEL_PATH / "best_model.pth",      # Change if your filename is different
    DEVICE
)

# Evaluation mode
model.eval()

print("Model Loaded Successfully")

Model Loaded Successfully


In [16]:
print(input_tensor.shape)

torch.Size([1, 3, 224, 224])


In [17]:
info = disease_db[predicted_class]

print("="*60)
print(f"Disease : {predicted_class}")
print(f"Confidence : {confidence*100:.2f}%")
print("="*60)

print("\nDescription")
print(info["description"])

print("\nSymptoms")
for item in info["symptoms"]:
    print("•", item)

print("\nCause")
for item in info["cause"]:
    print("•", item)

print("\nTreatment")
for item in info["treatment"]:
    print("•", item)

print("\nPrevention")
for item in info["prevention"]:
    print("•", item)

print("\nSeverity")
print(info["severity"])

print("\nFarmer Advisory")
print(info["farmer_advisory"])

Disease : Bacterialblight
Confidence : 98.48%

Description
A serious bacterial disease affecting rice leaves caused by Xanthomonas oryzae pv. oryzae.

Symptoms
• Yellowing leaf tips
• Water-soaked streaks
• Leaves dry from tip to base

Cause
• Bacterial infection
• Heavy rainfall
• Strong winds

Treatment
• Copper-based bactericide
• Balanced fertilizer application
• Remove severely infected plants

Prevention
• Use resistant rice varieties
• Avoid excess nitrogen fertilizer
• Maintain field sanitation

Severity
High

Farmer Advisory
Monitor the field regularly. Remove infected leaves early and avoid water stagnation.
